# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
"""One row = one client web page (content_id), same unit as w02. The starter file has no calendar dates, every number is a trailing window that ends on the export day:

    days 61-90 back      | days 31-60 back (prev_30d) | days 1-30 back (last_30d)
    <---------------- impressions_90d / clicks_90d / sessions_90d ------------------>

My label compares last_30d against prev_30d. So the honest "prediction moment" for my lane is day -30, the start of the last 30 days. Anything measured after that point belongs to the label window and cant be a feature. The queries below check the grain and that the windows actually nest like the picture says."""

from pathlib import Path
import duckdb

candidates = [
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("content_refresh_anonymized.csv"),
]
data_path = next(p for p in candidates if p.exists())
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW pages AS SELECT * FROM read_csv_auto('{data_path.as_posix()}')")

def q(sql):
    return con.sql(sql).df()

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_id) AS n_pages, COUNT(DISTINCT client_id) AS n_clients FROM pages"))

# grain probe: zero rows back = one row per page holds
display(q("SELECT content_id, COUNT(*) AS c FROM pages GROUP BY content_id HAVING c > 1 LIMIT 5"))

# window checks: every page is older than the 90 day window, day counters stay inside 0-90,
# and last_30d + prev_30d never exceed the 90 day total (so days 61-90 = 90d - last - prev is >= 0)
q("""
SELECT MIN(content_age_days) AS min_age_days,
       MAX(days_with_impressions) AS max_days_with_impr,
       MAX(days_with_sessions) AS max_days_with_sessions,
       SUM(CASE WHEN impressions_90d < impressions_last_30d + impressions_prev_30d THEN 1 ELSE 0 END) AS impr_windows_broken,
       SUM(CASE WHEN clicks_90d < clicks_last_30d + clicks_prev_30d THEN 1 ELSE 0 END) AS click_windows_broken,
       SUM(CASE WHEN sessions_90d < sessions_last_30d + sessions_prev_30d THEN 1 ELSE 0 END) AS session_windows_broken
FROM pages
""")

,n_rows,n_pages,n_clients
0,30000,30000,32


,content_id,c


,min_age_days,max_days_with_impr,max_days_with_sessions,impr_windows_broken,click_windows_broken,session_windows_broken
0,90,88,90,0.0,0.0,0.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
"""Sorting all 44 columns into the four buckets, using day -30 as the moment I predict.
feature = known before day -30. label = the target or what its built from. context = ids for grouping/splitting only. excluded = everything else, each with a reason.
The big call here is that all the 90 day totals and 90 day rates (ctr, avg_position, days_with_impressions...) are OUT, because a third of their window is the label window. The reference pipeline uses them, I dont. Instead I keep the two windows that sit fully before day -30: prev_30d (days 31-60) and days 61-90, which I back out as 90d - last_30d - prev_30d. The 90d columns are only used for that subtraction, never fed in raw."""

import pandas as pd

features = {
    "search_volume": "keyword demand estimate, content metadata",
    "competition": "keyword competition 0-1, content metadata",
    "competition_level": "LOW/MEDIUM/HIGH bucket of the same",
    "cpc": "keyword cost per click, content metadata",
    "content_type": "keyword / feedly / comparison article",
    "main_intent": "search intent of the target keyword",
    "word_count": "article length (measured at export, assumed mostly stable)",
    "content_age_days": "page age, shifted back 30 days to the prediction moment",
    "impressions_prev_30d": "impressions days 31-60, fully before day -30",
    "clicks_prev_30d": "clicks days 31-60",
    "sessions_prev_30d": "sessions days 31-60",
}
label = {
    "trend_direction": "label source: down = last_30d < 0.8 x prev_30d",
    "trend_pct": "the % change the direction is cut from",
    "impressions_last_30d": "the label window itself",
}
context = {
    "content_id": "page pseudonym, joins only",
    "client_id": "client pseudonym, grouped train/test split only",
}
excluded = {
    "impressions_90d": "includes the label window, only used to back out days 61-90",
    "clicks_90d": "includes the label window, only used to back out days 61-90",
    "sessions_90d": "includes the label window, only used to back out days 61-90",
    "clicks_last_30d": "measured inside the label window = future info",
    "sessions_last_30d": "measured inside the label window = future info",
    "pageviews_90d": "90d total incl. label window, no 30d split to back it out",
    "users_90d": "90d total incl. label window",
    "engaged_sessions_90d": "90d total incl. label window",
    "ai_sessions_90d": "90d total incl. label window",
    "scroll_events_90d": "90d total incl. label window",
    "days_with_impressions": "counted over 90d incl. label window",
    "days_with_sessions": "counted over 90d incl. label window",
    "ctr": "90d rate incl. label window (I rebuild a prev_30d ctr instead)",
    "avg_position": "90d average incl. label window",
    "engagement_rate": "90d rate incl. label window",
    "scroll_rate": "90d rate incl. label window",
    "ai_traffic_pct": "90d rate incl. label window",
    "impression_tier": "bucket of 90d impressions",
    "position_tier": "bucket of 90d avg_position",
    "days_since_last_update": "68% of pages were updated inside the label window, so at day -30 this value didnt exist yet",
    "freshness_tier": "bucket of days_since_last_update, same problem",
    "provider_used": "generation metadata, dictionary says not a model feature, ~71% blank",
    "model_used": "generation metadata, not a model feature",
    "char_count": "near duplicate of word_count",
    "char_count_tier": "bucket of char_count",
    "word_count_tier": "bucket of word_count, redundant",
    "age_tier": "bucket of content_age_days, redundant",
    "age_tier_order": "numeric version of age_tier, redundant",
}

contract = pd.DataFrame(
    [(col, bucket, why) for bucket, d in [("feature", features), ("label", label), ("context", context), ("excluded", excluded)] for col, why in d.items()],
    columns=["column", "bucket", "why"],
)
print(contract["bucket"].value_counts().to_string())
contract

bucket
excluded    28
feature     11
label        3
context      2


,column,bucket,why
0,search_volume,feature,"keyword demand estimate, content metadata"
1,competition,feature,"keyword competition 0-1, content metadata"
2,competition_level,feature,LOW/MEDIUM/HIGH bucket of the same
3,cpc,feature,"keyword cost per click, content metadata"
4,content_type,feature,keyword / feedly / comparison article
5,main_intent,feature,search intent of the target keyword
6,word_count,feature,"article length (measured at export, assumed mo..."
7,content_age_days,feature,"page age, shifted back 30 days to the predicti..."
8,impressions_prev_30d,feature,"impressions days 31-60, fully before day -30"
9,clicks_prev_30d,feature,clicks days 31-60


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
"""Every line of the contract gets a query here.
1) all 44 columns sit in exactly one bucket. 2) the label can be rebuilt from the window columns with zero mismatches, so I know exactly what its made of. 3) counts per client. 4) missing values, overall and per content_type, because the dictionary warns its patterned."""

all_cols = list(q("SELECT * FROM pages LIMIT 0").columns)
print("columns in file:", len(all_cols), "| in contract:", len(contract), "| duplicates:", contract["column"].duplicated().sum())
print("missing from contract:", sorted(set(all_cols) - set(contract["column"])), "| extra:", sorted(set(contract["column"]) - set(all_cols)))

# label rebuild: 'down' should be exactly last_30d < 0.8 * prev_30d
display(q("""
SELECT SUM(CASE WHEN trend_direction = 'down' AND NOT (impressions_last_30d < 0.8 * impressions_prev_30d) THEN 1 ELSE 0 END) AS down_but_rule_says_no,
       SUM(CASE WHEN trend_direction <> 'down' AND impressions_last_30d < 0.8 * impressions_prev_30d THEN 1 ELSE 0 END) AS rule_says_down_but_not_labelled,
       SUM(CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END) AS n_down,
       ROUND(AVG(CASE WHEN trend_direction = 'down' THEN 1.0 ELSE 0 END), 3) AS base_rate
FROM pages
"""))

# pages with zero prev_30d impressions can never be 'down' (nothing to drop from)
display(q("""
SELECT trend_direction, COUNT(*) AS n FROM pages WHERE impressions_prev_30d = 0 GROUP BY 1 ORDER BY n DESC
"""))

# rows per client: how lopsided is it
display(q("""
SELECT COUNT(*) AS clients, MIN(n) AS smallest_client, MEDIAN(n) AS median_client, MAX(n) AS biggest_client,
       ROUND(MAX(n) / SUM(n), 3) AS biggest_share,
       SUM(CASE WHEN n < 100 THEN 1 ELSE 0 END) AS clients_under_100_rows
FROM (SELECT client_id, COUNT(*) AS n FROM pages GROUP BY 1)
"""))

# missingness of the feature columns, split by content_type
q("""
SELECT content_type, COUNT(*) AS n,
       ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END), 3) AS search_volume_null,
       ROUND(AVG(CASE WHEN competition IS NULL THEN 1.0 ELSE 0 END), 3) AS competition_null,
       ROUND(AVG(CASE WHEN main_intent IS NULL THEN 1.0 ELSE 0 END), 3) AS intent_null,
       ROUND(AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END), 3) AS word_count_null,
       ROUND(AVG(CASE WHEN impressions_prev_30d IS NULL THEN 1.0 ELSE 0 END), 3) AS prev_impr_null
FROM pages GROUP BY 1 ORDER BY n DESC
""")

columns in file: 44 | in contract: 44 | duplicates: 0
missing from contract: [] | extra: []


,down_but_rule_says_no,rule_says_down_but_not_labelled,n_down,base_rate
0,0.0,0.0,16262.0,0.542


,trend_direction,n
0,new,2236
1,flat,1152


,clients,smallest_client,median_client,biggest_client,biggest_share,clients_under_100_rows
0,32,3,567.0,7008,0.234,8.0


,content_type,n,search_volume_null,competition_null,intent_null,word_count_null,prev_impr_null
0,keyword article,27207,0.014,0.014,0.01,0.283,0.0
1,feedly article,2096,1.000,1.000,1.00,0.000,0.0
2,comparison article,697,0.000,0.000,0.00,0.000,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
"""What this data can never tell me, with the number behind each one."""

lim = q("""
SELECT ROUND(AVG(CASE WHEN days_since_last_update < 30 THEN 1.0 ELSE 0 END), 3) AS updated_inside_label_window,
       ROUND(AVG(CASE WHEN impressions_prev_30d = 0 THEN 1.0 ELSE 0 END), 3) AS prev_window_empty,
       MIN(impressions_90d) AS min_impressions_90d,
       MIN(sessions_90d) AS min_sessions_90d,
       MIN(content_age_days) AS min_age_days
FROM pages
""")
display(lim)

print("""1. One snapshot, no dates. There is exactly one label window per page, so I cant test whether a pattern holds next month or in another season. A time-aware split is impossible on this file, the honest split I have is by client.
2. The panel is lopsided: the biggest client is about 23% of all rows and 8 clients have under 100 pages, so pooled numbers lean on a few clients.
3. The slice is pre-filtered: every page has impressions AND sessions and is at least 90 days old. Nothing here says anything about brand new pages or pages with zero traffic.
4. The label is a +-20% impressions threshold between two 30 day windows. Its a proxy for "this page is losing search visibility", not proof the page needs a refresh, and impressions are GSC only (no ranking or click quality in the label).
5. About 11% of pages had zero prev_30d impressions, so they can never be labelled down. They are easy negatives that make AUC look nicer than the ranking really is.
6. Window overlap: 68% of pages show an update date inside the label window, and all the 90 day totals include it. Thats why they are excluded above.
7. This contract is on the 30k starter slice. The full warehouse release (unbalanced client history, GSC-only early rows with zero-filled GA4, the 90 day query table overlapping the last months) is not used here, my lane runs on the same starter slice as w01/w02. So none of the warehouse checks are covered by this contract and any claim I make stays inside these 32 clients and this one window.""")

,updated_inside_label_window,prev_window_empty,min_impressions_90d,min_sessions_90d,min_age_days
0,0.683,0.113,1,1,90


1. One snapshot, no dates. There is exactly one label window per page, so I cant test whether a pattern holds next month or in another season. A time-aware split is impossible on this file, the honest split I have is by client.
2. The panel is lopsided: the biggest client is about 23% of all rows and 8 clients have under 100 pages, so pooled numbers lean on a few clients.
3. The slice is pre-filtered: every page has impressions AND sessions and is at least 90 days old. Nothing here says anything about brand new pages or pages with zero traffic.
4. The label is a +-20% impressions threshold between two 30 day windows. Its a proxy for "this page is losing search visibility", not proof the page needs a refresh, and impressions are GSC only (no ranking or click quality in the label).
5. About 11% of pages had zero prev_30d impressions, so they can never be labelled down. They are easy negatives that make AUC look nicer than the ranking really is.
6. Window overlap: 68% of pages show an upd

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.